# 06. Heterogeneidade e uplift funnel

Esta seção é uma extensão pós-confirmatória e exploratória. A S6 já encerrou a avaliação confirmatória no teste selado: o UpliftTree pré-selecionado não demonstrou vantagem confirmatória sobre o baseline de propensão (Delta Qini = -0.0088; IC95% [-0.0492, +0.0302]). Portanto, aqui não há reabertura do teste selado, nem resseleção retrospectiva de modelo, nem mudança da conclusão principal do projeto.

O objetivo é mais operacional: entender que perfis recebem maior score de uplift estimado em desenvolvimento e verificar se o ranking otimizado para `visit` parece carregar a mesma ordenação para `conversion` e `spend`.

## Índice

- [7.1 Dados permitidos](#s7-1)
- [7.2 Ranking exploratório de desenvolvimento](#s7-2)
- [7.3 Perfil do topo vs. base do ranking](#s7-3)
- [7.4 Outcomes observados por quantil](#s7-4)
- [7.5 Funnel uplift](#s7-5)
- [7.6 Concordância entre rankings](#s7-6)
- [7.7 Surrogate interpretável](#s7-7)
- [7.8 Artefatos da S7](#s7-8)
- [7.9 Sleeping Dogs e leitura final](#s7-9)

---

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = next(parent for parent in PROJECT_ROOT.parents if (parent / 'src').exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.config import POOLED_TREATMENT_COL, PROJECT_ROOT as CONFIG_PROJECT_ROOT, SEED
from src.data import add_pooled_treatment, load_hillstrom
from src.i18n import make_lang
from src.reports import build_s7_heterogeneity_report
from src.splits import get_train_val

PROJECT_ROOT = CONFIG_PROJECT_ROOT
lang = make_lang('pt')
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 140)
ARTIFACTS_S7_DIR = PROJECT_ROOT / 'artifacts' / 's7'
ARTIFACTS_S7_DIR.mkdir(parents=True, exist_ok=True)

Failed to import duecredit due to No module named 'duecredit'


<a id="s7-1"></a>

## 7.1 Dados permitidos

Usamos apenas treino e validação materializados pelos manifests persistidos. A mensagem impressa abaixo é uma checagem simples: o teste selado não é carregado neste notebook. A contagem de `conversion` no dataset completo é usada somente para dimensionar a raridade do desfecho, sem abrir outcomes/covariáveis do holdout selado por meio de `load_sealed_test`.

In [2]:
df = load_hillstrom()
df_pooled = add_pooled_treatment(df)
train_df, val_df = get_train_val(df_pooled)

print(f'train={len(train_df):,} | validation={len(val_df):,} | sealed test not loaded')
print(f"conversion positives in full dataset={int(df['conversion'].sum()):,} / {len(df):,} ({df['conversion'].mean():.2%})")

train=38,400 | validation=12,800 | sealed test not loaded
conversion positives in full dataset=578 / 64,000 (0.90%)


<a id="s7-2"></a>

## 7.2 Ranking exploratório de desenvolvimento

Para manter esta etapa barata e interpretável, ajustamos um X-learner com árvore rasa (`max_depth=4`) no treino e pontuamos a validação. Como o experimento pooled é randomizado, com dois braços tratados e um braço de controle, a propensão conhecida é P(T=1|X)=2/3 para todas as linhas. Esse valor é uma consequência do desenho experimental, não um hiperparâmetro ajustado; passá-lo explicitamente impede que o `causalml` estime um modelo auxiliar de propensão. A mesma especificação é repetida para `visit`, `conversion` e `spend`. Este ranking não substitui o modelo primário histórico da S6; ele é apenas uma ferramenta descritiva para perfis, quantis e funnel.

In [3]:
report = build_s7_heterogeneity_report(train_df, val_df, full_df=df)
profile = report['profile']
quantile_outcomes = report['quantile_outcomes']
funnel_metrics = report['funnel_metrics']
funnel_spearman = report['funnel_spearman']
surrogate = report['surrogate']

print(f"conversion positives={report['conversion_positives']:,} | rate={report['conversion_rate']:.2%}")

conversion positives=578 | rate=0.90%


<a id="s7-3"></a>

## 7.3 Perfil do topo vs. base do ranking

A tabela compara o quantil inferior e o quantil superior do score de uplift estimado para `visit`. Variáveis contínuas aparecem como médias; variáveis categóricas e binárias aparecem como participação do nível em cada extremo. Esta leitura descreve o grupo de alto uplift estimado, não identifica Persuadables observáveis individualmente e não confirma heterogeneidade causal formal.

In [4]:
profile.head(16).round(4)

,variable,level,bottom_quantile,top_quantile,delta_top_minus_bottom
0,history,mean,237.8625,378.4234,140.5610
1,recency,mean,6.8266,4.3777,-2.4488
2,womens,0,0.8535,0.0039,-0.8496
3,womens,1,0.1465,0.9961,0.8496
4,mens,0,0.0586,0.5887,0.5301
5,mens,1,0.9414,0.4113,-0.5301
6,channel,Web,0.7023,0.3883,-0.3141
7,history_segment,1) $0 - $100,0.4234,0.1559,-0.2676
8,zip_code,Rural,0.4348,0.1895,-0.2453
9,channel,Phone,0.2129,0.3965,0.1836


<a id="s7-4"></a>

## 7.4 Outcomes observados por quantil

Esta tabela organiza a validação por quantis do score de `visit` e mostra as médias observadas de `visit`, `conversion` e `spend`. A pergunta é se o topo do funil se move junto com outcomes mais próximos de receita. Se a ordenação melhora `visit` mas não melhora `spend`, a mensagem operacional é clara: otimizar para visit pode não otimizar para revenue.

In [5]:
quantile_outcomes.round(4)

,quantile,n,score_visit_mean,visit_mean,conversion_mean,spend_mean
0,1,2560,0.0165,0.1672,0.0090,1.1366
1,2,2560,0.0487,0.1305,0.0074,1.2344
2,3,2560,0.0634,0.1070,0.0070,0.7586
3,4,2560,0.0764,0.1375,0.0043,0.3075
4,5,2560,0.0959,0.1918,0.0164,2.0993


<a id="s7-5"></a>

## 7.5 Funnel uplift

Agora cada outcome recebe seu próprio ranking exploratório. `conversion` é especialmente difícil: há apenas 578 positivos em 64.000 linhas, cerca de 0,9% do dataset. Por isso, diferenças aparentes no fundo do funil devem ser lidas com cuidado; rankings divergentes sugerem que o proxy de topo de funil pode enganar, mas a amostra pode não ser suficiente para quantificar quanto com precisão.

In [6]:
funnel_metrics.round(4)

,outcome,qini_auc,uplift_auc,uplift_at_30pct,incremental_mean_top_30pct
0,visit,0.0615,0.0365,0.1024,0.0930
1,conversion,0.0216,0.0011,0.0039,0.0064
2,spend,NaN,NaN,NaN,0.8878


<a id="s7-6"></a>

## 7.6 Concordância entre rankings

A matriz de Spearman compara a ordenação induzida pelos scores de `visit`, `conversion` e `spend`. Correlações baixas ou instáveis reforçam a tese de funnel: o cliente mais promissor para visita não precisa ser o mesmo cliente mais promissor para compra ou valor gasto.

In [7]:
funnel_spearman.pivot(index='score_a', columns='score_b', values='spearman_corr').round(3)

score_b,conversion,spend,visit
score_a,,,
conversion,1.000,0.131,0.652
spend,0.131,1.000,0.094
visit,0.652,0.094,1.000


<a id="s7-7"></a>

## 7.7 Surrogate interpretável

A árvore abaixo é um surrogate exploratório treinado para aproximar quem cai no quantil superior do score de `visit`. O alvo aqui não é o outcome causal; é a etiqueta de alto ranking estimado. Portanto, as regras são uma forma compacta de descrever o ranking, não uma explicação causal confirmatória.

In [8]:
print(f"positive_rate={surrogate['positive_rate']:.2%}")
print(f"balanced_accuracy={surrogate['balanced_accuracy']:.3f}")
print(surrogate['rules'])

positive_rate=20.00%
balanced_accuracy=0.835
|--- num__womens <= 0.50
|   |--- num__history <= 693.66
|   |   |--- class: 0
|   |--- num__history >  693.66
|   |   |--- class: 0
|--- num__womens >  0.50
|   |--- num__mens <= 0.50
|   |   |--- num__recency <= 3.50
|   |   |   |--- class: 1
|   |   |--- num__recency >  3.50
|   |   |   |--- class: 0
|   |--- num__mens >  0.50
|   |   |--- num__history <= 802.15
|   |   |   |--- class: 1
|   |   |--- num__history >  802.15
|   |   |   |--- class: 1



<a id="s7-8"></a>

## 7.8 Artefatos da S7

Persistimos apenas tabelas derivadas de treino/validação para revisão e publicação. Nada em `artifacts/s6/` é lido para recalcular resultado, alterado ou sobrescrito.

In [9]:
profile.to_csv(ARTIFACTS_S7_DIR / 's7_top_bottom_profile.csv', index=False)
quantile_outcomes.to_csv(ARTIFACTS_S7_DIR / 's7_quantile_outcomes.csv', index=False)
funnel_metrics.to_csv(ARTIFACTS_S7_DIR / 's7_funnel_metrics.csv', index=False)
funnel_spearman.to_csv(ARTIFACTS_S7_DIR / 's7_funnel_spearman.csv', index=False)
print(f'S7 artifacts saved to {ARTIFACTS_S7_DIR.relative_to(PROJECT_ROOT)}')

S7 artifacts saved to artifacts\s7


<a id="s7-9"></a>

## 7.9 Sleeping Dogs e leitura final

O dataset não possui `unsubscribe`, opt-out, reclamação ou qualquer medida direta de dano. Assim, uplift negativo em `visit` é apenas um sinal fraco de possível resposta adversa, não evidência de Sleeping Dogs no sentido operacional forte.

A conclusão correta da S7 é limitada: há perfis e rankings exploratórios que ajudam a pensar em segmentação e no risco de usar `visit` como proxy de receita, mas a S6 permanece negativa para a hipótese confirmatória primária. A S7 não é uma porta dos fundos para declarar vitória de uplift modeling sobre response targeting.